# Document ProcessingPipeline

**Module 8 · Lesson 8.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Document Loaders - Extract Text from Any Format
- Three Chunking Strategies - Coded and Compared
- Measuring Chunk Quality - Retrieval Accuracy
- Production Document Pipeline - Complete Class
- PDF Parsing Ecosystem - 6 Tools, Tiered Strategy
- OCR for Scanned Documents - Tesseract, PaddleOCR, Surya, Cloud

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy pymupdf pymupdf4llm beautifulsoup4 markitdown python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Pipeline Nobody Budgeted For

> 💡 **Info**
>
> A fintech team ships a RAG assistant in one sprint. The demo corpus: 40 clean Markdown files. Retrieval accuracy: beautiful. Then compliance hands them the real corpus - 40,000 PDFs: two-column policy documents, scanned annexures from 2011, spreadsheets embedded in Word files, and Hindi circulars typed in a font from 1998. Accuracy craters. The postmortem finds nothing wrong with the LLM, the embeddings, or the vector database. **Every failure traced back to ingestion:** text extracted in the wrong order, tables shredded into word soup, scans that silently produced empty strings, and Hindi that copy-pasted as gibberish. This lesson is about the unglamorous stage that decides whether RAG works at all: turning real-world documents into clean, retrievable chunks.

### 🎯 What you will be able to do after this lesson

- **Build** a document ingestion pipeline in pure Python: load PDF/HTML/Markdown, clean, chunk, and batch-embed with gemini-embedding-001.

- **Compare** fixed, recursive, and semantic chunking on the same corpus and measure which retrieves best - instead of trusting defaults.

- **Evaluate** the 2026 PDF-parsing and OCR tool tiers (PyMuPDF4LLM, Docling, Marker, MinerU, Tesseract, PaddleOCR, vision models) and pick per document type.

- **Design** an India-ready pipeline that classifies each PDF as true-Unicode, legacy-font, or scanned - and routes it accordingly.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lesson 8.1 (the five-step RAG pipeline and embeddings), a Google AI Studio API key in `GOOGLE_API_KEY`, and `pip install google-genai numpy pymupdf beautifulsoup4`. Every code block runs standalone in under a minute.

## 60-Second Warm-Up: What You Already Know

Three recalls from Lesson 8.1 - all load-bearing today. Click a card to check yourself.

> 🍳 **Analogy**
>
> **Chunking is like cutting ingredients for cooking.** Cut too large - uneven cooking, some raw, some burnt. Cut too small - turns to mush. Cut just right, *matching the shape of the ingredient* - perfect dish. **Fixed chunking = cutting everything to 2cm cubes. Semantic chunking = cutting along the natural grain of the meat.**

> 💡 **Info**
>
> ⚠️ Misconception: "PDF text extraction is a solved problem"
> Engineers assume every PDF library returns the same text and pick whichever installed first - that is the anti-pattern to avoid. A PDF stores *positioned glyphs*, not paragraphs: reading order, columns, tables, headers, and footers are all reconstructed by the parser, and different parsers reconstruct them differently. On clean single-column PDFs the differences are cosmetic; on two-column layouts, financial tables, or scans they are catastrophic - and silent. The parser never throws "I shredded your table"; it just returns plausible-looking wrong text that poisons every downstream step. **Rule: verify extraction quality per document type before trusting any tool.**

Document Ingestion Pipeline

## Build One: The Ingestion Pipeline in Pure Python

Steps 1-4 build a working pipeline with zero frameworks: a loader for four formats, three chunkers, a measurement harness that scores them, and a production-shaped class that ties it together. Type these out - the intuition you build here is what makes the tool survey in steps 5-10 make sense.

---

## 🎯 Concept 1: Document Loaders - Extract Text from Any Format

### Document Loaders - Extract Text from Any Format

PDF, HTML, Markdown, plain text - one loader class to handle them all.

**Why start here:** before you can chunk anything, you need text - and every format hides it differently. PDFs bury it in positioned glyphs, HTML wraps it in navigation and scripts, Markdown decorates it with syntax. A loader's job is to give the rest of the pipeline one consistent input: clean plain text (or better, Markdown) plus metadata about where it came from.

> 📦 **Analogy**
>
> **A loader is a mailroom clerk.** Envelopes arrive in every shape - couriered boxes (PDF), padded mailers (HTML), postcards (Markdown) - and the clerk opens each with the right technique, then places the CONTENTS flat on the same tray for processing. One tray, many envelope types: that is the `DocumentLoader` class, where `load()` looks at the extension and dispatches to the right opener.
> **Where the analogy breaks down:** a real clerk preserves the letter exactly. Text extraction is lossy - flattening a PDF to plain text destroys headings, table structure, and emphasis. That loss is fine for prose and fatal for tables, which is why production loaders (step 5) emit structured Markdown instead of flat text.

A PDF, an HTML page, and a Markdown file hold the same article. Which needs the LEAST work to extract clean text?

**📝 `01_loaders.py`** - *Loaders*

In [ ]:
# DOCUMENT LOADERS - Extract text from any format
# pip install pymupdf beautifulsoup4
import re
from pathlib import Path

class DocumentLoader:
    """Load and extract text from PDF, HTML, Markdown, or plain text."""

    def load(self, path: str) -> dict:
        ext = Path(path).suffix.lower()
        loaders = {".pdf": self._pdf, ".html": self._html,
                   ".md": self._markdown, ".txt": self._text}
        loader = loaders.get(ext, self._text)
        text = loader(path)
        text = re.sub(r'[ \t]+', ' ', text)   # collapse spaces ONLY
        text = re.sub(r'\n{3,}', '\n\n', text).strip()  # keep paragraph breaks for the recursive chunker
        return {"source": path, "text": text, "words": len(text.split()), "format": ext}

    def _pdf(self, path):
        import pymupdf  # "import fitz" is the legacy alias
        doc = pymupdf.open(path)
        return " ".join(page.get_text() for page in doc)

    def _html(self, path):
        from bs4 import BeautifulSoup
        with open(path, encoding="utf-8") as f: soup = BeautifulSoup(f.read(), "html.parser")
        [s.decompose() for s in soup(["script","style","nav","footer"])]
        return soup.get_text(" ")

    def _markdown(self, path):
        with open(path, encoding="utf-8") as f: text = f.read()
        text = re.sub(r'#{1,6}\s+', '', text)  # remove headers markup
        text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)  # links
        text = re.sub(r'[*_`~]+', '', text)  # formatting
        return text

    def _text(self, path):
        with open(path, encoding="utf-8") as f: return f.read()

# Demo with simulated files
loader = DocumentLoader()
print("DocumentLoader supports: .pdf, .html, .md, .txt")
print("  PDF: pymupdf extracts text, tables, even images-as-text")
print("  HTML: BeautifulSoup strips scripts/nav/footer, keeps content")
print("  MD: regex removes #headers, [links](url), **bold**")
print("  All output: clean plaintext ready for chunking")

# Output:
#   DocumentLoader supports: .pdf, .html, .md, .txt
#   PDF: pymupdf extracts text, tables, even images-as-text
#   HTML: BeautifulSoup strips scripts/nav/footer, keeps content
#   MD: regex removes #headers, [links](url), **bold**
#   All output: clean plaintext ready for chunking

- Scene 1: the illusion - a two-column PDF page looks perfectly readable to a human.

- Scene 2: the X-ray - the file actually stores floating glyph boxes with (x,y) coordinates: no paragraphs, no columns, no table structure.

- Scene 3: naive extraction reads straight across BOTH columns, interleaving sentences and swallowing the footer - garbled output, red X.

- Scene 4: a layout-aware loader detects the column blocks, orders them, drops the footer, and emits clean Markdown - green check.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Scene buttons jump; the caption narrates. Garbage at Load poisons every later stage.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

What just happened? One class, four formats, one output contract. Two production details hide in the small print: **encoding="utf-8"** on every open() - Windows defaults to cp1252 and CRASHES on the first Devanagari character (the exact documents step 10 handles) - and the unknown-extension fallback to _text, which silently pretends binaries are text. **The tradeoff:** flat text is perfect for chunking prose but destroys table structure; production loaders (step 5) emit Markdown to keep both.

---

## 🎯 Concept 2: Three Chunking Strategies - Coded and Compared

### Three Chunking Strategies - Coded and Compared

Fixed-size, recursive (split by structure), and semantic (split by meaning).

**The decision this step teaches:** where to cut. Lesson 8.1 showed why chunks must exist; this step codes the three standard ways to draw the boundaries - by count (fixed), by structure (recursive), by meaning (semantic) - so you can see the actual chunks each one produces from the same document.

> 🎬 **Analogy**
>
> **Chunking strategies are three film editors cutting the same movie into clips.** The first cuts every 10 minutes exactly, mid-dialogue if needed (fixed-size). The second cuts only at scene changes - a new location, a new conversation (recursive: paragraph, then sentence boundaries). The third watches the STORY and cuts when the plot line shifts, even mid-scene (semantic: embedding similarity between neighboring sentences).
> **Where the analogy breaks down:** movie scenes have hard boundaries; topics in text blur into each other. Semantic chunking needs a similarity threshold to decide "new topic", and a badly tuned threshold produces confetti - dozens of three-word fragments that retrieve nothing.

The fixed-size chunker cuts every 200 words, overlap 50. What failure is GUARANTEED to happen eventually?

**📝 `02_chunking_strategies.py 3`** - *Strategies*

In [ ]:
# THREE CHUNKING STRATEGIES - Coded and compared
import re
import numpy as np

# ── Strategy 1: Fixed-size chunks ──
def chunk_fixed(text, size=200, overlap=50):
    """Split every N words with overlap. Simple but dumb."""
    assert 0 < overlap < size, "need 0 < overlap < size"
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        c = " ".join(words[i:i+size])
        if len(c.split()) >= 20: chunks.append(c)
        elif chunks: chunks[-1] += " " + c  # merge short tail - never drop text
    return chunks

# ── Strategy 2: Recursive (structure-aware) ──
def chunk_recursive(text, max_size=200, separators=None):
    """Split by paragraph, then sentence, then word. Respects structure."""
    separators = separators or ["\n\n", "\n", ". ", " "]
    chunks = []
    def _split(text, sep_idx=0):
        if len(text.split()) <= max_size:
            if len(text.split()) >= 15: chunks.append(text.strip())
            return
        if sep_idx >= len(separators):
            chunks.append(" ".join(text.split()[:max_size]))
            return
        parts = text.split(separators[sep_idx])
        current = ""
        for part in parts:
            if len((current + part).split()) <= max_size:
                current += separators[sep_idx] + part
            else:
                if current.strip(): _split(current.strip(), sep_idx + 1)
                current = part
        if current.strip(): _split(current.strip(), sep_idx + 1)
    _split(text)
    return chunks

# ── Strategy 3: Semantic (meaning-aware) ──
def chunk_semantic(text, threshold=0.22):
    """Group sentences by similarity. Word-overlap (Jaccard) is the cheap
    proxy here; production uses sentence embeddings + cosine (Lesson 8.1)."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.split())>=3]
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        prev_words = set(" ".join(current[-2:]).lower().split())
        curr_words = set(sentences[i].lower().split())
        overlap = len(prev_words & curr_words) / max(len(prev_words | curr_words), 1)
        if overlap >= threshold:  # same topic (Jaccard cutoff)
            current.append(sentences[i])
        else:  # topic shift = new chunk
            chunks.append(" ".join(current))
            current = [sentences[i]]
    if current: chunks.append(" ".join(current))
    return chunks

# ── Compare all 3 on the same document ──
doc = """Netsetos is an educational technology company founded in 2024 in Hyderabad.
We offer courses in Generative AI, Data Science, and Cloud Computing.

Our refund policy allows full refunds within 7 days of purchase.
After 7 days, we offer a 50 percent refund up to 30 days.
After 30 days, no refunds are available.

The GenAI course costs 14999 rupees and includes lifetime access.
Students also get Discord community access and weekly live sessions.
Google Cloud credits worth 5000 rupees are included.

Live classes are held daily at 7 PM IST on YouTube.
Recordings are uploaded within 2 hours after the session.
The course completion rate is 85 percent with 4.8 star rating."""

print("Chunking Strategy Comparison:\n")
for name, fn in [("Fixed (60w)", lambda t: chunk_fixed(t,60,10)),
                  ("Recursive", lambda t: chunk_recursive(t,60)),
                  ("Semantic", lambda t: chunk_semantic(t))]:
    chunks = fn(doc)
    print(f"  {name}: {len(chunks)} chunks")
    for i,c in enumerate(chunks):
        print(f"    [{i}] ({len(c.split())}w) {c[:65]}...")
    print()

# Output: (abridged)
#   Chunking Strategy Comparison:
#   Fixed (60w): 5 chunks   [0] (60w) Netsetos is an educational technology company...
#   Recursive: 4 chunks     [0] (21w) Netsetos is an educational technology company...
#   Semantic: 7 chunks      [0] (35w) Netsetos is an educational technology company...

- Scene 1: the document - refund policy, pricing, FAQ paragraphs, with one sentence spanning a topic shift.

- Scene 2: fixed-size - the ruler cuts every N words, splitting "the refund is 50" from "percent within 30 days" mid-sentence (red).

- Scene 3: recursive - cuts land ON paragraph boundaries first, sentences second: same size budget, smarter cut points.

- Scene 4: semantic - a similarity line dips exactly at the topic shift; the cut lands in the valley.

- Scene 5: the payoff - one query runs against all three chunk sets: fixed misses, recursive and semantic hit; the scoreboard settles it.

Controls: Play/Pause, Reset, speed. Replay scene 5 - measurement, not taste, picks the strategy.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? Three strategies on the same document: **Fixed** split every 60 words, cutting mid-paragraph. **Recursive** split by \n\n first, keeping paragraphs intact - refund policy stays in one chunk. **Semantic** grouped sentences by topic overlap - each chunk is about one subject. **Recursive is the best default for most RAG systems. Semantic gives highest quality but is slower (needs embeddings).**

---

## 🎯 Concept 3: Measuring Chunk Quality - Retrieval Accuracy

### Measuring Chunk Quality - Retrieval Accuracy

Good chunks = good retrieval. We measure: does the right chunk get retrieved for each question?

**The habit this step installs:** never argue about chunking in the abstract. Write down real questions users will ask, note which chunk holds each answer, and score every strategy on "did the right chunk rank first?". Twenty lines of code turn a taste debate into a measurement.

> 📋 **Analogy**
>
> **This is a find-it drill for filing systems.** Two colleagues organize the same office: one files by document size, one by topic. You do not debate which cabinet looks tidier - you hand an intern five real requests ("find the refund policy") and time how often they pull the right folder first try. The filing scheme that wins the drill wins the office.
> **Where the analogy breaks down:** our drill has four questions - a smoke test, not a benchmark. Production evaluation (Lesson 8.11) uses dozens of labeled queries per corpus and proper metrics (recall@k, MRR) before declaring a winner.

Before running step 3: which strategy will win the retrieval test on our structured FAQ document?

**Naming the metric:** what we compute below is **recall@1** (also called hit rate): for each test question, does the single top-ranked chunk contain the answer? With 4 questions and 3 correct top hits, recall@1 = 0.75. The exercises extend this to **recall@5** (is the answer anywhere in the top five?). Our "does the expected keyword appear" check is a cheap proxy for a human-labeled relevant chunk - good enough to compare strategies, not a publishable benchmark.

**📝 `03_chunk_eval.py`** - *Evaluation*

In [ ]:
# CHUNKING EVALUATION - Which strategy retrieves best?
# pip install google-genai numpy
# Requires chunk_fixed / chunk_recursive / chunk_semantic and doc from 02_chunking_strategies.py
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
EMBED = "gemini-embedding-001"

def embed_batch(texts, task):
    """ONE API call for many texts - never loop per chunk."""
    r = client.models.embed_content(
        model=EMBED, contents=texts,
        config=types.EmbedContentConfig(task_type=task))
    return [np.array(e.values) for e in r.embeddings]

def eval_retrieval(chunks, test_cases):
    """Does the correct chunk rank #1 for each question?"""
    embs = embed_batch(chunks, "RETRIEVAL_DOCUMENT")
    qs = embed_batch([q for q, _ in test_cases], "RETRIEVAL_QUERY")
    correct = 0
    for qe, (question, expected_keyword) in zip(qs, test_cases):
        scores = [(i, float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e))))
                  for i, e in enumerate(embs)]
        scores.sort(key=lambda x: -x[1])
        if expected_keyword.lower() in chunks[scores[0][0]].lower():
            correct += 1
    return correct / len(test_cases)

# Test questions with expected keywords in the correct chunk
test_cases = [
    ("What is the refund policy?", "refund"),
    ("How much does the course cost?", "14999"),
    ("When are live classes?", "7 PM"),
    ("What is the completion rate?", "85"),
]

print("Retrieval Accuracy by Chunking Strategy:\n")
for name, chunks in [
    ("Fixed", chunk_fixed(doc, 60, 10)),
    ("Recursive", chunk_recursive(doc, 60)),
    ("Semantic", chunk_semantic(doc)),
]:
    acc = eval_retrieval(chunks, test_cases)
    print(f"  {name:12s}: {acc*100:.0f}% " + "█" * int(acc * 20))

# Output: (your corpus WILL differ - that is exactly why you measure)
#   Retrieval Accuracy by Chunking Strategy:
#   Fixed       : 50% ██████████
#   Recursive   : 100% ████████████████████
#   Semantic    : 75% ███████████████

#### 💡 What just happened

What just happened? Twenty lines turned "which chunking is best?" from an opinion into a number: embed chunks as documents, embed questions as queries (different `task_type` - recall Lesson 8.1), rank by cosine, score rank-1 hits. That is recall@1. **When to trust it:** as a relative comparison between strategies on YOUR corpus - not as an absolute quality score; four questions is a smoke test, and keyword-hit is a proxy for real labels.

---

## 🎯 Concept 4: Production Document Pipeline - Complete Class

### Production Document Pipeline - Complete Class

Load any format, chunk with chosen strategy, embed, return ready-to-index chunks.

**What changes here:** steps 1-3 were separate experiments. Production needs one object with one contract: text goes in, indexed-ready chunk records come out - each with an id, the text, its embedding, and metadata. This class is that contract, plus the two hygiene steps (cleaning, batched embedding) that separate demo code from deployable code.

> 🏭 **Analogy**
>
> **The pipeline class is an assembly line with fixed stations.** Raw material enters (text), each station does one job in order - clean, chunk, embed, label - and an inspected, boxed unit exits with a serial number (`netsetos_faq_0`) and a spec sheet (metadata). Nobody hand-carries parts between stations; the conveyor (the `process()` method) guarantees the order.
> **Where the analogy breaks down:** this line shows the happy path only. A real factory line has reject bins and alarms - retry queues, malformed-file handling, incremental re-indexing when a document updates. Those arrive in Lesson 12.x and the exercises.

The pipeline is about to embed 500 chunks. How many API calls should that take?

**📝 `04_pipeline.py`** - *Production*

In [ ]:
# COMPLETE DOCUMENT PROCESSING PIPELINE
# Requires the chunk_* functions and doc from 02_chunking_strategies.py
from google import genai
from google.genai import types
import re

client = genai.Client()
EMBED = "gemini-embedding-001"

class DocumentPipeline:
    """Load → Clean → Chunk → Embed. Ready for a vector DB."""

    def __init__(self, strategy="recursive", chunk_size=200, overlap=50):
        assert overlap < chunk_size, "overlap must be smaller than chunk_size"
        self.strategy = strategy
        self.chunk_size = chunk_size
        self.overlap = overlap

    def _clean(self, text):
        text = re.sub(r'[ \t]+', ' ', text)   # spaces only - NEVER eat \n\n,
        text = re.sub(r'\n{3,}', '\n\n', text)  # or recursive chunking goes blind
        text = re.sub(r'Page \d+ of \d+', '', text)  # strip page furniture
        text = re.sub(r'http\S+', '', text)  # strip bare URLs
        return text.strip()

    def process(self, text, source="doc"):
        text = self._clean(text)
        if self.strategy == "fixed":
            chunks = chunk_fixed(text, self.chunk_size, self.overlap)
        elif self.strategy == "recursive":
            chunks = chunk_recursive(text, self.chunk_size)
        else:
            chunks = chunk_semantic(text)

        # ONE batched call embeds every chunk - a per-chunk loop is the
        # classic beginner mistake: slower, rate-limited, same bill
        r = client.models.embed_content(
            model=EMBED, contents=chunks,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
        return [{"id": f"{source}_{i}", "text": c,
                 "embedding": list(r.embeddings[i].values),
                 "source": source, "words": len(c.split())}
                for i, c in enumerate(chunks)]

pipe = DocumentPipeline(strategy="recursive", chunk_size=80)
results = pipe.process(doc, source="netsetos_faq")
print(f"Pipeline: {len(results)} chunks, ready for vector DB")
for r in results:
    print(f"  {r['id']:20s} {r['words']:3d}w  {r['text'][:50]}...")

# Output:
#   Pipeline: 4 chunks, ready for vector DB
#   netsetos_faq_0        58w  Netsetos is an educational technology company fo...
#   netsetos_faq_1        42w  Our refund policy allows full refunds within 7 d...
#   netsetos_faq_2        45w  The GenAI course costs 14999 rupees and includes...
#   netsetos_faq_3        40w  Live classes are held daily at 7 PM IST on YouTu...

#### 💡 What just happened

What just happened? The pipeline glued the pieces into one contract: clean (preserving paragraph breaks so the recursive chunker can see them), chunk by strategy, embed EVERYTHING in one batched call, and emit records with id + text + embedding + metadata. **The two invariants worth memorizing:** the cleaner must never destroy the structure the chunker depends on, and embedding calls should be batched, never looped. For production re-ingestion, swap positional ids for content hashes (step 9) so re-runs upsert instead of duplicate.

## Go Deeper: Production Document Processing

The from-scratch pipeline handles clean text. Production corpora are not clean. Steps 5-10 survey the 2026 tool landscape - PDF parsers, OCR, non-PDF formats, advanced chunking, metadata, and the India-specific pipeline - so you know which tool to reach for when the happy path fails.

---

## 🎯 Concept 5: PDF Parsing Ecosystem - 6 Tools, Tiered Strategy

### PDF Parsing Ecosystem - 6 Tools, Tiered Strategy

PyMuPDF4LLM, pdfplumber, Docling, Marker, Unstructured, VLMs. When to use which.

**Why a survey, not one tool:** our from-scratch loader called one library and hoped. Production corpora contain clean reports AND scanned contracts AND 200-page tables - no single parser wins all three. The 2026 ecosystem has settled into tiers: fast text-layer extractors, layout-aware models, and vision-language models. Knowing the tiers saves you from both under-engineering (shredded tables) and over-engineering (a GPU cluster to parse memos).

> 🔧 **Analogy**
>
> **Pick parsers like you pick repair tools.** A pocket screwdriver (PyMuPDF4LLM) handles 80 percent of jobs in seconds and lives in every drawer. A workshop bench (Docling, Marker, MinerU) handles complex jobs - tables, formulas, weird layouts - at the cost of setup and compute. A specialist contractor (vision-language models, cloud OCR) is called for the jobs nothing else can do, and billed accordingly. The skill is triage: sending every memo to the contractor bankrupts you; fixing a gearbox with a pocket screwdriver strands you.
> **Where the analogy breaks down:** a wrong tool on a bolt visibly fails. A wrong parser on a PDF SILENTLY returns wrong text - so the tiering must be verified with spot checks per document type, not assumed.

A fast parser just returned plausible-looking text from a complex financial PDF. What is the hidden risk?

| Tool | Speed/page | Tables | License | Best For |
|---|---|---|---|---|
| PyMuPDF4LLM | ~0.12s | ✅ Markdown | AGPL/Comm | Default - born-digital, high throughput |
| pdfplumber | ~0.10s | ✅ strong on gridded tables | MIT | Tables with visible grids, visual debug |
| Docling(IBM) | Moderate | ✅ top table accuracy | MIT | Air-gapped, complex layouts, huge community |
| Marker | ~11s | ✅ ML | GPL | Scientific papers, equations (LaTeX) |
| Unstructured | ~1.3s | ✅ Hi-Res | Apache/Comm | Enterprise, 70+ file types, Auto routing |
| VLMs | ~2-10s | ✅ Best | API | Degraded scans, charts (~200× more expensive) |

> ✅ **Info**
>
> 🏁 Production tiered parsing strategy
> **1.** PyMuPDF4LLM for fast native extraction (fractions of a paisa per page). **2.** Quality check: text length, structure detection. **3.** Escalate failed/complex pages to Docling or Marker. **4.** VLMs (Qwen2.5-VL, Gemini 2.5) as final resort for degraded scans (roughly a hundred times the tier-1 cost). **5.** Post-process: convert tables to natural language for better retrieval. Key insight: **domain determines accuracy more than parser choice** - 55-point gap between easy/hard domains dwarfs tool differences.

#### 💡 What just happened

What just happened? No single PDF parser wins everywhere. The tiered strategy routes documents through increasingly expensive tools based on quality. **[PyMuPDF4LLM](https://github.com/pymupdf/pymupdf4llm) handles the large majority of born-digital PDFs** at negligible cost. Docling's MIT license and air-gapped capability make it the enterprise choice. Marker wins for scientific papers. Unstructured's Auto strategy routes per-page automatically. VLMs are the nuclear option - best quality, highest cost.

**📝 `05_tiered_parse.py`** - *language-python*

*Illustrative Tier-1 parse snippet (requires an `annual_report.pdf` on disk) - shown for reference, not run here:*

```python
# Tier 1 in four lines - and the quality gate that decides escalation
# pip install pymupdf4llm
import pymupdf4llm

md = pymupdf4llm.to_markdown("annual_report.pdf")  # headings, tables -> Markdown

# crude quality gate: near-empty output means scanned or broken -> escalate
if len(md.strip()) < 200:
    print("ESCALATE: no text layer - route to OCR (step 6) or Docling")
else:
    print(f"Tier 1 OK: {len(md)} chars of Markdown")

# Output:
#   Tier 1 OK: 48213 chars of Markdown
```


---

## 🎯 Concept 6: OCR for Scanned Documents - Tesseract, PaddleOCR, Surya, Cloud

### OCR for Scanned Documents - Tesseract, PaddleOCR, Surya, Cloud

Preprocessing pipeline. Open-source vs cloud. Sarvam Vision for Indian scripts.

**The mental switch:** a scanned PDF contains no text at all - it is a photograph of a page. Everything you know about extraction stops applying; you are now doing computer vision. The surprise of this step: OCR accuracy is decided mostly BEFORE the OCR engine runs, in preprocessing.

> 🖼️ **Analogy**
>
> **OCR is restoring an old photograph before identifying the faces.** The photo arrives tilted in the frame, dusty, and faded. You straighten it (deskew), wipe the dust (denoise), and boost the contrast (binarize) - and only THEN does the face become recognizable. Skip the restoration and even the best expert misidentifies people; do it well and a cheap tool succeeds.
> **Where the analogy breaks down:** restoration cannot invent what was never captured. A 100-dpi scan of smudged carbon paper has lost information permanently - the fix is rescanning at higher resolution or a human transcriber, not a better model.

You run text extraction on a scanned PDF. What comes back?

| OCR Engine | Languages | Best For | Cost |
|---|---|---|---|
| Tesseract v5 | 100+ | Baseline, CPU-only | Free |
| PaddleOCR v5 | 109 | Production (clear step up from v4) | Free |
| Surya OCR | 90+ | PDF-native, used in Marker | Free (GPL) |
| Azure Doc Intelligence | 300+ | Enterprise, top-tier accuracy | ~$1.50/1K pages |
| Sarvam Vision | 22 (Indian) | Indian scripts, benchmark leader | Paid API + free tier |

> 📦 **Info**
>
> 📸 OCR preprocessing is the biggest accuracy lever
> The critical pipeline: **rescale to 300 DPI → grayscale → denoise (cv2.medianBlur) → binarize (Otsu's method - auto-picks the black/white cutoffing) → deskew**. Use adaptive thresholding for uneven lighting. For Indian scripts: Sarvam Vision outperforms Gemini 3 Pro, Claude Opus 4.5, and GPT-5.2 on Indic OCR benchmarks spanning 22 Indian languages. PaddleOCR's PP-OCRv5 is the best free alternative for Hindi - a clear step up from v4 on Indic benchmarks.

#### 💡 What just happened

What just happened? OCR accuracy depends more on **preprocessing** than on the OCR engine itself. Always preprocess scanned documents before feeding to any OCR engine. **PaddleOCR v5** is the best free option (about 109 languages, clearly better than v4). For Indian scripts, **Sarvam Vision** is the reported leader: it scored 84.3 on olmOCR-Bench (Feb 2026), ahead of Gemini 3 Pro (80.2) and GPT 5.2 (69.8), across 22 Indian languages. **The tradeoff:** open-source engines are free and private but need preprocessing skill; cloud APIs and VLMs buy accuracy on hard scans with money and data-residency questions - pick per document sensitivity and volume. Cloud services (Azure, Google, AWS) are best for enterprise scale at ~$1.50/1K pages.

- Scene 1: the problem - a tilted, speckled scan; text extraction returns 0 characters (there IS no text layer).

- Scene 2: deskew - the baseline angle is detected and the page rotates straight (8.2 degrees to 0).

- Scene 3: denoise + binarize - speckles fade, the page snaps to crisp black-on-white.

- Scene 4: OCR sweeps line by line with per-word confidence - and preprocessing bought more accuracy than switching engines would.

Controls: Play/Pause, Reset, speed. The tier choice (Tesseract/PaddleOCR vs cloud vs VLM) comes AFTER preprocessing.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Non-PDF Formats - HTML, DOCX, Excel, Email, Multimodal

### Non-PDF Formats - HTML, DOCX, Excel, Email, Multimodal

Web crawling (Firecrawl, Crawl4AI). Word docs. ColPali for visual RAG.

**Widening the intake:** real corpora are maybe half PDF. The rest arrives as web pages, Word documents, spreadsheets, email threads, and slides - each with its own extraction quirks. The unifying trick of 2026 tooling: convert EVERYTHING to Markdown, because it is the one format that survives with structure intact and that LLMs read natively.

> 🚢 **Analogy**
>
> **Ingestion is a customs checkpoint at a port.** Cargo arrives in every container type - ships (websites), trucks (DOCX), courier bags (email) - and customs unpacks each onto the same standard inspection table: Markdown. One table, standard forms (metadata), whatever the container. That is why tools from Firecrawl to MarkItDown all converge on Markdown output.
> **Where the analogy breaks down:** some cargo cannot be unpacked into text at all - dashboards, infographics, engineering drawings. For those, visual retrieval (ColPali-family models that embed the page IMAGE directly) skips the unpacking entirely - the survey's last row.

HTML, DOCX, Excel, email - what single format do 2026 ingestion tools convert them all INTO?

| Format | Tool | Key Feature |
|---|---|---|
| HTML/Web | Trafilatura (F1=0.958) | Best content extraction, built-in dedup |
| Web crawling | Firecrawl / Crawl4AI | JS rendering, recursive crawl, → Markdown |
| DOCX | mammoth → markdownify | Semantic HTML → Markdown conversion |
| Excel/CSV | pandas → df.to_markdown() | LLMs understand Markdown tables natively |
| Email | email (stdlib) / extract-msg | EML + Outlook .msg support |
| PowerPoint | python-pptx | Shapes, text frames, speaker notes |
| Multimodal | ColPali (ICLR 2025) | Direct image → embedding, bypasses OCR |

#### 💡 What just happened

What just happened? Every format needs a specialized tool. The universal pattern: **convert to Markdown first, then chunk**. For web content, Trafilatura leads on quality (F1=0.958). For crawling JS-heavy sites, Crawl4AI (Apache 2.0, 50K⭐) or Firecrawl. **ColPali** is revolutionary: embeds page images directly without OCR → chunk → embed pipeline, outperforming text-based methods on visually complex documents. For traditional multimodal: extract images → VLM description → index text descriptions.

**📝 `07_markitdown.py`** - *language-python*

*Illustrative MarkItDown snippet (requires `policy.docx`, `pricing.xlsx`, `faq.html` on disk) - shown for reference, not run here:*

```python
# One tool, many formats -> the same Markdown inspection table
# pip install markitdown
from markitdown import MarkItDown

md = MarkItDown()
for f in ["policy.docx", "pricing.xlsx", "faq.html"]:
    result = md.convert(f)
    print(f, "->", len(result.text_content), "chars of Markdown")

# Output:
#   policy.docx -> 18240 chars of Markdown
#   pricing.xlsx -> 2456 chars of Markdown
#   faq.html -> 9102 chars of Markdown
```


---

## 🎯 Concept 8: Advanced Chunking - Contextual, Late, Proposition, Agentic

### Advanced Chunking - Contextual, Late, Proposition, Agentic

Beyond recursive splitting. LLM-enhanced chunks. Embed-first chunking.

**When defaults stop being enough:** recursive chunking loses CONTEXT at every cut - "the fee is 2 percent" in chunk 47 no longer says which fee, which product, which year. The four techniques here all attack that one problem (context loss at boundaries) from different angles, at increasing cost. Reach for them only after step 3's measurement shows naive chunking actually failing.

> 🔖 **Analogy**
>
> **Contextual chunking is a librarian stamping every photocopied page.** Tear a page out of a book and it loses its spine: which book? which chapter? The librarian stamps each copy - "Netsetos Refund Policy, section 3, 2026 edition" - before filing (an LLM prepends a situating sentence to each chunk). Late chunking is the librarian who reads the WHOLE book first, then files each page while remembering the full plot (embed the full document, then split the token embeddings).
> **Where the analogy breaks down:** stamps are cheap; contextual chunking costs one LLM call per chunk, and late chunking demands a long-context embedding model. The librarian's judgment is free - these are not.

Chunk 47 says only: "the fee is 2 percent". What did the chunking cut destroy?

> 📦 **Info**
>
> 🧰 Advanced chunking strategies
> - **Contextual ([Anthropic, Sep 2024](https://www.anthropic.com/news/contextual-retrieval)):** an LLM prepends a short situating sentence to each chunk before embedding. Anthropic reports a large drop in retrieval failures; the one-time cost is modest with prompt caching. Highest-ROI advanced technique.
> - **Late Chunking (Jina AI):** Embed entire document first with long-context model, then split embeddings. Each chunk carries full document context via self-attention. No extra LLM calls. Limited by embedding model context window (~10 pages).
> - **Proposition-based (Dense X Retrieval):** decompose passages into atomic self-contained facts. Notably better recall on factoid questions, but too fragmented for general use - reserve it for medical/legal fact retrieval.
> - **Agentic:** LLM decides chunk boundaries based on topic shifts and completeness - cents per document, the priciest option. Best for high-value documents (contracts, medical records).
> - **Structure-aware:** MarkdownHeaderTextSplitter preserves heading metadata on each chunk. Combine with RecursiveCharacterTextSplitter for oversized sections.

#### 💡 What just happened

What just happened?**Chunking configuration impacts retrieval quality as much as embedding model selection.** Most teams over-invest in embeddings and under-invest in chunking. Start with recursive ~512-token chunks - the measured baseline winner in independent benchmarks. If you need more: contextual chunking is the highest ROI (modest one-time cost, large reported failure reduction). Late chunking is cheap but needs a long-context embedding model. Proposition chunking is precision-focused. Agentic is expensive but handles complex documents. **Semantic chunking underperformed recursive splitting in independent benchmarks (tiny-fragment problem) - measure before adopting.**

**📝 `08_contextual_chunk.py`** - *language-python*

In [ ]:
# Contextual chunking in one function: stamp each chunk with its origin
def contextualize(chunk, doc_summary):
    prompt = (f"Document: {doc_summary}\n\nChunk: {chunk}\n\n"
              "Write ONE sentence situating this chunk in the document. Sentence only.")
    ctx = client.models.generate_content(model="gemini-2.5-flash",
                                         contents=prompt).text.strip()
    return ctx + " " + chunk  # prepend, THEN embed

print(contextualize("The fee is 2 percent.",
                     "Netsetos 2026 refund and fees policy"))

# Output:
#   This sentence states the processing fee in the Netsetos 2026 refund
#   and fees policy. The fee is 2 percent.

---

## 🎯 Concept 9: Metadata Extraction & Text Cleaning - Enrichment, Dedup, Normalization

### Metadata Extraction & Text Cleaning - Enrichment, Dedup, Normalization

LLM-generated metadata. MinHash dedup. Unicode normalization. Pre-filtering.

**The quiet multiplier:** two chunks with identical text can behave completely differently at query time if one carries `{"source": "policy-2026.pdf", "section": "refunds", "date": "2026-03"}` and the other carries nothing. Metadata enables filtered retrieval ("only 2026 policies"), citations, and freshness ranking - and cleaning/dedup stops near-identical chunks from crowding out diverse results.

> 🥛 **Analogy**
>
> **Metadata is the label on every container in a shared office fridge.** Unlabeled containers make lunch retrieval a gamble; "Priya - dal - Monday" makes it instant, lets you filter ("anything from this week?"), and lets the Friday cleanup (dedup) throw out the three identical forgotten curries instead of someone's fresh lunch.
> **Where the analogy breaks down:** fridge labels help any reader. Metadata only pays off if your retrieval system actually FILTERS on it - a vector store queried without metadata filters ignores the labels entirely. Label at ingestion, filter at query: both halves or neither works.

The same document was ingested twice by accident. What goes wrong at query time?

> ✅ **Info**
>
> 📈 Metadata is the most underrated RAG improvement
> - **Basic metadata:** source filename, page number, section headers, dates, authors. Extract automatically during parsing.
> - **LLM-generated metadata (MetaRAG):** topic classification, entities, summaries, user intents. Published evaluations report meaningfully higher precision than content-only retrieval.
> - **Pre-filtering:** Always filter by metadata BEFORE vector search. "Q4 refund policy" without date filter returns outdated 2023 version (cosine 0.91) over correct 2025 doc (0.87).
> - **Deduplication:** exact duplicates fall to content hashing (one line - see below); NEAR-duplicates need MinHash + LSH via the datasketch library (MinHash fingerprints text so near-identical chunks get near-identical signatures; LSH buckets signatures for fast lookup without comparing every pair). Vector databases are adding native MinHash indexing for dedup at scale.
> - **Text cleaning:** Strip headers/footers, boilerplate, zero-width characters (\u200b, \u200c, \u200d). Language detection: FastText (120K sentences/sec, 176 languages).
> - **Unicode normalization:** Apply NFC form to ALL text. Critical for Indic scripts where same word has multiple valid codepoints.

#### 💡 What just happened

What just happened? Metadata enrichment is the **single most important performance driver for RAG** according to C3 AI's production experience. Without metadata filtering, vector search returns semantically similar but contextually wrong documents. LLM-generated metadata (topic, entities, intent) adds 9+ percentage points of precision. Deduplication prevents wasted embedding budget on repeated content. Unicode NFC normalization is non-negotiable for multilingual pipelines.

**📝 `09_dedup_metadata.py`** - *language-python*

In [ ]:
# The two cheapest wins in this step: content-hash ids + a real metadata record
import hashlib, unicodedata

def make_record(chunk, source, page, section):
    chunk = unicodedata.normalize("NFC", chunk)  # one canonical byte form
    cid = hashlib.sha256(chunk.encode()).hexdigest()[:16]  # same text -> same id
    return {"id": cid, "text": chunk, "source": source,
            "page": page, "section": section}

r1 = make_record("Full refund within 7 days.", "policy.pdf", 3, "refunds")
r2 = make_record("Full refund within 7 days.", "policy_v2.pdf", 1, "refunds")
print(r1["id"] == r2["id"])  # re-ingestion upserts instead of duplicating

# Output:
#   True

---

## 🎯 Concept 10: India Document Processing - Legacy Fonts, Sarvam Vision, Devanagari Normalization

### India Document Processing - Legacy Fonts, Sarvam Vision, Devanagari Normalization

Kruti Dev detection. Font2Unicode. Mixed Hindi-English. Government PDFs.

**Why India gets its own step:** Indian document pipelines hit three failure modes that English-only pipelines never see - legacy fonts that LOOK like Hindi but store Roman bytes, one word with multiple valid Unicode encodings, and Hindi-English code-switching inside a single paragraph. Miss any of the three and retrieval silently returns nothing for Hindi queries.

> 🎭 **Analogy**
>
> **A legacy-font PDF is text wearing a costume.** On screen it is flawless Devanagari - but underneath, the bytes are Roman letters ("dk;kZy;"), and the FONT is a costume that paints क over the letter "d". Copy-paste rips the costume off and you see the Roman body. Every downstream step - embedding, search, matching - works on the body, not the costume, so the pipeline must re-dress the text properly (convert to real Unicode) before anything else.
> **Where the analogy breaks down:** costumes are standardized; legacy fonts are not. Each font family (Kruti Dev, Chanakya, Shivaji) uses a DIFFERENT glyph mapping, so the converter must match the font - a Kruti Dev table applied to Chanakya text produces different gibberish.

A government Hindi PDF renders perfectly on screen. Copy-paste gives "dk;kZy;". Your first hypothesis?

> 📦 **Info**
>
> 🇮🇳 India-specific document processing
> - **Legacy font problem:** dozens of legacy Devanagari font families (Kruti Dev 010, Chanakya, Shivaji01) - each with 180+ glyph-to-codepoint mappings - store what LOOKS like Hindi as Roman codepoints. "क" stored as "d". Looks correct with font installed, gibberish on copy-paste. Pervasive in government, NCERT textbooks, newspapers.
> - **Detection:** Extract text → check ratio of ASCII to Devanagari characters. High ASCII + no Devanagari in a "Hindi" PDF = legacy font.
> - **Conversion:** kru2uni (LTRC/IIIT Hyderabad) for Kruti Dev. Font2Unicode (Punjabi University) for Gurmukhi. Convert before any processing.
> - **Sarvam Vision:** 3B-param VLM covering 22 Indian languages; outscored Gemini 3 Pro and GPT 5.2 on olmOCR-Bench (Feb 2026). Handles mixed-script, low-quality scans, stamps, hand-filled fields. Paid API with a free tier - check current per-page pricing.
> - **Devanagari normalization:** Apply NFC + Indic NLP Library normalizer. Same word can have multiple valid Unicode representations (क़ = U+0958 or U+0915+U+093C). Without normalization, identical words fail to match.
> - **Mixed Hindi-English:** Language detection per chunk → route to appropriate embedding model → transliterate Romanized Hindi (IndicXlit) → normalize.

#### 💡 What just happened

What just happened? India document processing has three unique challenges that don't exist in English pipelines: **legacy fonts** (invisible problem - text looks fine with font installed), **Devanagari normalization** (same word, multiple codepoints → retrieval failures), and **mixed-script content** (Hindi-English code-switching in 250M users). The pipeline: classify PDF (Unicode/legacy/scanned) → appropriate extraction → Devanagari normalization → metadata enrichment → embedding. Always test with real Indian government PDFs - they're the hardest case. **When to invest:** if under a few percent of your corpus is legacy-font or scanned Hindi, manual conversion may beat building the full detection pipeline - measure the mix first.

- Scene 1: looks perfect - a government PDF renders beautiful Devanagari.

- Scene 2: copy-paste shock - the clipboard holds Roman gibberish ("dk;kZy;"); the font was painting Devanagari glyphs over Roman codepoints.

- Scene 3: detection - a "Hindi" doc that is 95 percent ASCII with zero Devanagari codepoints trips the legacy-font alarm.

- Scene 4: fix + normalize - the converter maps legacy codepoints to real Unicode, then NFC merges alternate encodings of the same syllable so words finally match in retrieval.

Controls: Play/Pause, Reset, speed. Classify every PDF first: true-Unicode, legacy-font, or scanned.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🔗 Where this goes next (Module 8)
> - **In Lesson 8.3 (Embeddings & Vector Stores)** the chunks you produced here get a real home - production vector databases with ANN indexes instead of a Python list, and we'll come back to `task_type` and Matryoshka dimensions in depth.
> - **In Lesson 8.4 (Advanced Retrieval)**, chunk quality meets query quality: hybrid search and reranking sit ON TOP of the ingestion you built today - garbage chunks cap what they can rescue.
> - **In Lesson 8.9 (Multimodal RAG)**, the ColPali thread from step 7 becomes a full lesson - retrieving page IMAGES when text extraction is the wrong tool entirely. **Lesson 8.11 (RAG Evaluation)** upgrades step 3's recall@1 smoke test into a real evaluation harness.

## Key Takeaways

> 📦 **Info**
>
> - **RAG quality is decided at ingestion.** Load, clean, and chunk BEFORE any retrieval magic - garbage in the pipeline poisons every downstream step.
> - **Chunking is a measurable decision, not a default.** Fixed, recursive, and semantic chunking give different retrieval accuracy on the same corpus - measure recall on your own documents.
> - **PDF extraction is a tiered problem.** Fast text-layer tools for clean PDFs, layout models (Docling, MinerU) for complex ones, OCR and vision models only for scans.
> - **Metadata and cleaning pay compound interest.** Source, page, section, dates, and dedup enable filtered retrieval and stop near-duplicate chunks from crowding results.
> - **Indian documents need one extra stage.** Classify each PDF first: true Unicode, legacy font (convert), or scanned (OCR) - then normalize Devanagari before embedding.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each maps to a pipeline stage you just built or surveyed.

---

## 🎓 Summary

You've completed the practical part of **Document ProcessingPipeline**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.2.html` - regenerate this notebook whenever the source HTML is updated.*
